# 03 — Score & baselines (Colab)

Rescore existing plan-executor predictions or run **programmatic baselines** without reloading the LLM.

Restores pilot data and results from Drive (notebook **02**).

In [ ]:
# @title Configuration
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
BRANCH = "main"  # @param {type:"string"}

WORK = "/content"
MUSICINSTRUCT = f"{WORK}/musicinstruct"
RUN_ID = "pilot_v1"  # @param {type:"string"}
SPLIT = "test"  # @param ["test", "validation"]

USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/musicinstruct"
DRIVE_DATA = f"{DRIVE_ROOT}/data/pilot"
DRIVE_RESULTS = f"{DRIVE_ROOT}/runs/{RUN_ID}/plan_executor_llama"
LOCAL_RESULTS = f"{MUSICINSTRUCT}/results/plan_executor_llama"

RUN_BASELINES = True  # @param {type:"boolean"}
RESCORE_PLAN_EXECUTOR = True  # @param {type:"boolean"}

In [ ]:
# @title Bootstrap & restore from Drive
import os
import shutil
import subprocess
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")

if Path(MUSICINSTRUCT).is_dir():
    shutil.rmtree(MUSICINSTRUCT)
subprocess.run(
    f"git clone --branch {BRANCH} --depth 1 {MUSICINSTRUCT_REPO} {MUSICINSTRUCT}",
    shell=True,
    check=True,
)
os.chdir(MUSICINSTRUCT)
subprocess.run("pip install -q -e .", shell=True, check=True)

if USE_DRIVE:
    shutil.copytree(DRIVE_DATA, Path(MUSICINSTRUCT) / "data/pilot", dirs_exist_ok=True)
    shutil.copytree(DRIVE_RESULTS, Path(LOCAL_RESULTS), dirs_exist_ok=True)

manifest = f"{MUSICINSTRUCT}/data/pilot/pilot.jsonl"
preds = f"{LOCAL_RESULTS}/predictions.jsonl"
print("manifest:", manifest)
print("predictions:", preds, "exists:", Path(preds).is_file())

In [ ]:
# @title Rescore plan-executor predictions
import json
from pathlib import Path

if RESCORE_PLAN_EXECUTOR:
    out = Path(LOCAL_RESULTS) / "score_report.json"
    !musicinstruct score data/pilot/pilot.jsonl {LOCAL_RESULTS}/predictions.jsonl --split {SPLIT} --output {out}
    print(json.dumps(json.loads(out.read_text()).get("overall", {}), indent=2))
else:
    print("Skipped rescore")

In [ ]:
# @title Run programmatic baselines
import json
from pathlib import Path

if RUN_BASELINES:
    baseline_dir = Path(MUSICINSTRUCT) / "results/baselines"
    !musicinstruct run-baselines data/pilot/pilot.jsonl --output-dir {baseline_dir} --split {SPLIT}
    summary_path = baseline_dir / "baseline_report.json"
    if summary_path.is_file():
        data = json.loads(summary_path.read_text())
        for name, block in data.get("baselines", {}).items():
            print(name, block.get("overall"))
    if USE_DRIVE:
        import shutil

        drive_baselines = Path(DRIVE_ROOT) / f"runs/{RUN_ID}/baselines"
        shutil.copytree(baseline_dir, drive_baselines, dirs_exist_ok=True)
        print("Synced baselines →", drive_baselines)
else:
    print("Skipped baselines")